# 05.01 图邻接表与并行 SSSP 章节概述

本章节从有向带权图、单源最短路径以及图邻接表、 CSR 与转置 CSR 等基础知识出发，在 NPU 上实现并行 Bellman-Ford 算法，以解决单源最短路径（SSSP）问题。

## 1. 本章前置要求

学习本章前，建议先具备以下基础：

1. 编程基础：能够阅读和运行 Python、C/C++ 代码，理解数组、循环、函数和文件读写。
2. 数据结构基础：理解图中的顶点、边、边权、路径和有向图含义，知道邻接表或稀疏矩阵可以用于表示图。
3. 算法基础：了解最短路径问题的基本目标，知道 Dijkstra 或 Bellman-Ford 是求解最短路径的经典算法。
4. 实验环境基础：能够使用 Jupyter Notebook 运行 Code Cell，并对 CANN（Compute Architecture for Neural Networks，异构计算架构）、Ascend C 和 NPU（Neural Processing Unit，神经网络处理器）算子开发流程有初步认识。

## 2. 本章主要内容

本章围绕“图邻接表向 CSR 稀疏张量的格式转换”展开，并进一步使用转换后的 CSR 数据完成并行 SSSP（Single-Source Shortest Path，单源最短路径）算子设计。课程先从图邻接表这种直观表示出发，将其整理为连续数组形式的出边 CSR，再构造适合 Pull 松弛的转置 CSR；随后推导同步 Pull Bellman-Ford 松弛方法，并将其映射到 Atlas 910B3/A2 的多核与 UB（Unified Buffer，统一缓冲区）分块计算；最后完成测试数据生成、Ascend C Kernel、C++ Host、工程编译、NPU 运行、CPU Golden 对比和综合练习。

### 2.1 单源最短路径

对带权有向图 $G=(V,E)$ 和源点 $s$，SSSP（Single-Source Shortest Path，单源最短路径）计算 $s$ 到每个顶点 $v$ 的最小路径权重 $d[v]$。本实验只接受有限、非负的 `float32` 边权，因此 CPU 标杆使用 Dijkstra；NPU 侧选择更容易并行化的同步 Bellman-Ford。

松弛是核心操作：若 `dist[u] + weight(u,v) < dist[v]`，就用更短候选值更新 `dist[v]`。

### 2.2 图邻接表、CSR 与转置 CSR

邻接表通常用“每个源顶点对应一组出边”的方式描述图，例如 `adj[u]=[(v,w), ...]` 表示从顶点 `u` 出发可以到达若干目标顶点 `v`，边权为 `w`。这种表示便于理解和插入边，但每个顶点的边列表长短不一，不适合直接作为 NPU Kernel 的连续输入。

CSR（Compressed Sparse Row，压缩稀疏行）把邻接表压缩成三段连续数组：`row_ptr[V+1]` 记录各顶点出边的起止位置，`col_idx[E]` 保存边指向的目标顶点，`weights[E]` 保存对应边权。为了让 Pull 模型按目标顶点读取入边，本实验还会从出边 CSR 构造转置 CSR：每行对应目标顶点 `v`，索引数组 `src_idx[E]` 保存所有能到达 `v` 的前驱顶点 `u`。

In [ ]:
import numpy as np

# 邻接表：adj[u] 保存从源顶点 u 出发的所有 (目标顶点, 边权)
adj = {
    0: [(1, 2.0), (2, 5.0)],
    1: [(2, 1.0), (3, 4.0)],
    2: [(3, 1.0)],
    3: [],
}
vertex_count = len(adj)

# 第一步：邻接表转换为出边 CSR，得到 row_ptr / col_idx / weights
row_ptr, col_idx, weights = [0], [], []
for u in range(vertex_count):
    for v, w in adj[u]:
        col_idx.append(v)
        weights.append(w)
    row_ptr.append(len(col_idx))
print("outgoing row_ptr:", row_ptr)
print("outgoing col_idx:", col_idx)
print("outgoing weights :", weights)

# 第二步：为了 Pull 松弛，再把出边组织转换为按目标顶点排列的转置 CSR
incoming = [[] for _ in range(vertex_count)]
for u in range(vertex_count):
    for edge_id in range(row_ptr[u], row_ptr[u + 1]):
        v = col_idx[edge_id]
        incoming[v].append((u, weights[edge_id]))

in_row_ptr, src_idx, in_weights = [0], [], []
for row in incoming:
    row.sort()
    src_idx.extend(u for u, _ in row)
    in_weights.extend(w for _, w in row)
    in_row_ptr.append(len(src_idx))
print("incoming row_ptr:", in_row_ptr)
print("incoming src_idx:", src_idx)
print("incoming weights :", in_weights)

![CSR 与转置 CSR](images/csr_and_transpose.png)

### 2.3 同步 Pull Bellman-Ford

出边 Push 模型会让多个核心同时尝试更新同一目标顶点的距离 `dist_out[v]`，容易产生写冲突。本实验采用 Pull 模型：每个核心负责一组目标顶点，读取其所有入边，在本地计算最小距离后连续写回，从而避免跨核竞争。

下图展示了一轮同步 Pull 松弛过程。每个核心先从只读的 `dist_in` 获取上一轮距离，再通过转置 CSR 找到目标顶点 v 的所有前驱及边权，计算 `min(dist_in[u]+w)`，最后由唯一负责该顶点的核心写入 `dist_out[v]`，因此不会发生跨核写冲突。

![同步 Pull 松弛](images/pull_relaxation.png)

单轮同步公式：

$$d^{(k+1)}[v]=\min\left(d^{(k)}[v],\min_{(u,v)\in E}(d^{(k)}[u]+w(u,v))\right).$$

每一轮计算都只读取上一轮保存的距离 dist_in=d^(k)，并将本轮得到的新距离写入另一个数组 dist_out=d^(k+1)，避免新旧结果相互干扰。等所有核心完成本轮计算后，再交换两个数组，进入下一轮。由于包含 V 个顶点的最短路径最多经过 V-1 条边，因此最多执行 V-1 轮。这种交替读写两个数组的方式称为双缓冲。

In [ ]:
INF = np.float32(1e30)
def pull_once(row_ptr, src_idx, weights, dist_in):
    out = np.array(dist_in, dtype=np.float32, copy=True)
    for v in range(len(row_ptr)-1):
        for e in range(row_ptr[v], row_ptr[v+1]):
            u = src_idx[e]
            if dist_in[u] < INF * .5:
                out[v] = min(out[v], dist_in[u] + weights[e])
    return out

dist = np.array([0, INF, INF, INF], np.float32)
for k in range(3):
    dist = pull_once(row_ptr, src_idx, weights, dist)
    print(k + 1, dist)

### 2.4 NPU 映射与 UB 预算

Host 根据顶点数 V（Vertex Count，顶点数量）和可用 Vector Core（向量计算核心）数量确定实际启用的核心数，然后把连续目标顶点平均分配给各核心。每个核心先将完整的 `dist_in`（上一轮距离数组）搬入 UB（Unified Buffer，统一缓冲区），再把顶点按每批 256 个 `VERTEX_TILE=256` 处理，入边按每批 1024 条 `EDGE_TILE=1024` 处理，避免一次搬入过多数据。

当 `MAX_VERTEX_NUM=16384` 时，FP32 距离数组需要 `16384×4=64KB`。每批 256 个顶点需要读取 257 个行偏移，按 32 字节对齐后占 1056B；1024 条边的源顶点索引和权重共占约 8KB；256 个输出距离占约 1KB。主要数据合计约 74KB，能够放入 910B3 每核 192KB 的 UB。

下图展示了 ARM（处理器架构）Host 与 NPU 的协作过程。Host 负责初始化 ACL（Ascend Computing Language，昇腾计算语言运行接口）、申请 GM（Global Memory，全局内存）、循环启动 Kernel（NPU 核函数）、交换距离缓冲区并取回结果。多个 Core（计算核心）分别处理不同顶点范围，共同读取 CSR、`dist_in`（输入距离） 和 `dist_out`（输出距离）。

![Host 与 NPU 数据流](images/npu_dataflow.png)

### 2.5 复杂度与边界

单轮计算需要遍历全部 V（顶点数）和 E（边数），因此时间复杂度为 $O(V+E)$。由于最短路径最多经过 $V-1$ 条边，算法最坏执行 $V-1$ 轮，总复杂度约为 $O(V(V+E))$。如果某个顶点无法从源点到达，就用很大的 FP32 数值 `1e30` 表示“无穷远”，它不是实际距离，只是不可达状态的标记。


### 2.6 课后练习

1. `row_ptr[v+1]-row_ptr[v]` 表示什么？
2. Pull 为什么不需要跨核原子写？
3. 原地更新会破坏什么语义？

运行下面的 Code Cell 可以查看参考答案。


In [ ]:
from pathlib import Path

p = Path("answer/05.01_chapter_intro/answers.md")
if not p.exists():
    p = Path("05_图邻接表向CSR稀疏张量的格式转换") / p
print(p.read_text(encoding="utf-8"))


## 3. 学习目标

完成本章后，学习者能够：

1. 说明 SSSP、Bellman-Ford、图邻接表、CSR 与转置 CSR 之间的关系。
2. 将有向带权图转换为适合 Pull 松弛的转置 CSR 数据。
3. 理解顶点分核、UB 分块和双缓冲在 NPU 算子中的作用。
4. 使用 Ascend C 和 C++ 完成并行 SSSP Kernel 开发。
5. 使用 CPU Dijkstra 结果验证 NPU 输出正确性，并分析不可达顶点、零权边和边界规模等情况。

## 4. 小节简介与跳转链接

<table
  align="left"
  style="width: 80%;
         max-width: 1200px;
         margin: 0 auto 0 0 !important;
         text-align: left;">
  <thead>
    <tr>
      <th style="text-align: left;">小节</th>
      <th style="text-align: left;">简介</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;"><a href="./05.01_chapter_intro.ipynb" target="_self">05.01 图邻接表与并行 SSSP 章节概述</a></td>
      <td style="text-align: left;">介绍本章前置要求、主要内容，包括邻接表、CSR、转置 CSR、同步 Pull、UB 预算和实验边界，以及学习目标和小节导航。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><a href="./05.02_parallel_sssp.ipynb" target="_self">05.02 基于图邻接表向 CSR 转换的并行 SSSP 算子开发</a></td>
      <td style="text-align: left;">通过可执行 Code Cell 完成邻接表建图、CSR/转置 CSR 数据生成、Tiling、Ascend C Kernel、C++ Host、CMake 构建、NPU 运行和精度验证。</td>
    </tr>
    <tr>
      <td style="text-align: left;"><a href="./05.03_chapter_practice.ipynb" target="_self">05.03 章节实践</a></td>
      <td style="text-align: left;">通过补全单轮 Pull 松弛、计算多核 Tiling 参数，加深对算法与硬件映射的理解。</td>
    </tr>
  </tbody>
</table>
<div style="clear: both;"></div>